In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import time
import serial
import serial.tools.list_ports

In [228]:
img = cv2.imread(R'D:\Downloads\genshin2.jpg')
# img = cv2.imread(R"D:\Pictures\jinitaimei.png")
# img = cv2.imread(R"D:\Downloads\koji.jpg")
# img = cv2.imread(R"D:\Downloads\chunping.jpg")

In [230]:
gimg = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
gimg = cv2.resize(gimg,dsize= None,fx = 0.25,fy = 0.25)
# cimg = cv2.Canny(gimg, 50, 200)
cimg = cv2.Canny(gimg, 100, 400)
# cimg = cv2.Canny(gimg, 10, 100)

# roi = gimg[100:500, 400:900]
# roi = cv2.resize(roi,dsize= None,fx = 2,fy = 2)
# cimg = cv2.Canny(roi, 100, 200)

cv2.imshow("cimg",cimg)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [247]:
goto = lambda x:print(x)
for i in list(serial.tools.list_ports.comports()):
    print(i)
serial_port = serial.Serial("COM6", 115200, timeout=1)

COM6 - USB-SERIAL CH340 (COM6)


In [248]:
def zdt_move_abs(serial:serial.Serial,addr:int, direction:int, vel:int, acc:int, clk:int, raF:bool, snF:bool):
    global serial_port
    cmd = []
    cmd[0:13] = [0] * 13  # 初始化命令数组
    cmd = np.array(cmd, dtype=np.uint8)  # 转换为无符

    cmd[0]  =  addr                       # 地址
    cmd[1]  =  0xFD                       # 功能码
    cmd[2]  =  direction                        # 方向
    cmd[3]  =  0xFF & (vel >> 8)          # 速度(RPM)高8位字节
    cmd[4]  =  0xFF & (vel >> 0)          # 速度(RPM)低8位字节 
    cmd[5]  =  acc  & 0xFF                      # 加速度，注意：0是直接启动
    cmd[6]  =  0xFF & (clk >> 24)         # 脉冲数(bit24 - bit31)
    cmd[7]  =  0xFF & (clk >> 16)         # 脉冲数(bit16 - bit23)
    cmd[8]  =  0xFF & (clk >> 8)          # 脉冲数(bit8  - bit15)
    cmd[9]  =  0xFF & (clk >> 0)          # 脉冲数(bit0  - bit7 )
    cmd[10] =  int(raF)                       # 相位/绝对标志，false为相对运动，true为绝对值运动
    cmd[11] =  int(snF)                        # 多机同步运动标志，false为不启用，true为启用
    cmd[12] =  0x6B                       # 校验字节
    # print(list(cmd))
    # serial.write([0x02,0x36,0x6b])
    serial.write(list(cmd))
    

def zdt_parse_receive(serial,addr:list[int]):
    r = serial.read(1)
    # while r:
    #     if r in addr:
    
def zdt_goto(serial:serial.Serial,yx):
    _x = np.arctan(yx[1]/400)*4 / (np.pi*2) *3200 
    _y = np.arctan(yx[0]/400)*4 / (np.pi*2) *3200
    zdt_move_abs(serial,1,1,1000,253,int(np.round(_x,0)),True,False)
    time.sleep(0.01)
    zdt_move_abs(serial,2,1,1000,253,int(np.round(_y,0)),True,False)
    # print(np.round(_x))
    # print(np.round(_y))
    time.sleep(0.01)
    


In [249]:
# zdt_move_abs(serial_port,2,1,20,1,3200,True,False)
zdt_goto(serial_port,[500,500])
# np.arctan(100/1000)*4 / (np.pi*2) *3200 


In [250]:
zdt_goto(serial_port,[0,0])

In [191]:
serial_port.read(1)

b'\x01'

In [196]:
def check_bounds(yx:list[int], img_shape):
    yx[0] = max(0, min(yx[0], img_shape[0]))
    yx[1] = max(0, min(yx[1], img_shape[1]))
    return yx

In [252]:
bimg = cimg.copy()
blackboard = np.zeros_like(bimg)
# current_pos = bimg.shape[0] // 2, bimg.shape[1] // 2
current_pos = 0,0
point_num = np.count_nonzero(bimg)
display(point_num)
imgshape = bimg.shape
for i in range(point_num):
    found = False
    roi = None
    leftup = np.array([])
    j = 1
    while not found:
        leftup = [current_pos[0] - j , current_pos[1] - j]
        rightdown = [current_pos[0]+j , current_pos[1] + j]
        leftup = check_bounds(leftup,imgshape)
        rightdown = check_bounds(rightdown,imgshape)
        # print(leftup,rightdown)
        roi = bimg[leftup[0]:rightdown[0],leftup[1]:rightdown[1]]
        # print(roi)
        found = np.count_nonzero(roi)
        # print(found)
        j +=1
    in_roi = np.where(roi == 255)[0][0], np.where(roi == 255)[1][0]
    next_pos = tuple(leftup+np.array(in_roi))
    bimg[next_pos] = 0
    current_pos = next_pos
    if i%2 == 0:
        zdt_goto(serial_port,next_pos)
        blackboard[next_pos] = 255
        cv2.imshow('bimg', blackboard)
        # cv2.imshow('bimg', bimg)
        if cv2.waitKey(1) == ord("q"):
            break
    
zdt_goto(serial_port,[0,0])
cv2.waitKey(0)
cv2.destroyAllWindows()
    

2295

In [246]:
cv2.destroyAllWindows()
serial_port.close()